# State Management

**Module:** 14 — AI Orchestration

What belongs in state, checkpoints/resume, and schema evolution.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Decide what to put in agent/workflow state vs external stores
- Implement checkpoint and resume semantics
- Evolve state schemas safely


## What Belongs in State?

### Definition
State is the durable, structured snapshot a run needs to continue correctly — messages, cursors, intermediate artifacts references, flags.

### Why it matters
Too little state → cannot resume. Too much → huge checkpoints, PII sprawl, slow writes.

### How it works
Store references to large blobs (object storage keys), not megabyte payloads. Separate secrets. Prefer typed schemas.

### Intuition
A bookmark + outline, not a photocopy of the whole library each step.

### Pitfalls
- Putting raw PDFs in state
- Duplicating the entire CRM record every turn
- Unvalidated dict soup

### When to use
All durable workflows and agents.


| Put in state | Put outside state |
|--------------|-------------------|
| Step cursor / status | Large documents (S3/GCS) |
| Small structured slots | Vector indexes |
| Human approval flags | Long-term user memory (Module 13) |
| Tool idempotency keys | Secrets manager values |
| Error last_reason | Analytics event lakes |

```
State = working_set + pointers + control_flags
```


In [ ]:
# Demo 1: state with pointers instead of blobs
state = {
    "schema_version": 1,
    "status": "running",
    "messages": [{"role": "user", "content": "summarize doc"}],
    "artifacts": {"doc_uri": "s3://bucket/doc-123.pdf", "bytes": None},
    "flags": {"human_approved": False},
}
assert state["artifacts"]["bytes"] is None
print(state["artifacts"]["doc_uri"])


## Checkpoints & Resume

### Definition
A checkpoint is a persisted state snapshot at a safe point; resume loads it and continues the graph/chain.

### Why it matters
Process crashes, deploys, and HITL waits require resume — otherwise work is lost.

### How it works
Write checkpoint after each successful node (or transactionally with side effects). On start, load latest checkpoint by `thread_id`/`run_id`.

### Intuition
Save game files for agents.

### Pitfalls
- Checkpointing mid-non-idempotent side effect
- Resuming wrong version
- Not fencing concurrent workers on same run

### When to use
Long graphs, HITL, any run > a few seconds.


In [ ]:
# Demo 2: checkpoint store with optimistic versioning
from copy import deepcopy

class CheckpointStore:
    def __init__(self):
        self.data = {}  # run_id -> (version, state)
    def save(self, run_id, state, expected_version=None):
        cur = self.data.get(run_id)
        ver = 0 if cur is None else cur[0]
        if expected_version is not None and expected_version != ver:
            raise ConflictError("stale checkpoint")
        self.data[run_id] = (ver + 1, deepcopy(state))
        return ver + 1
    def load(self, run_id):
        return self.data.get(run_id)

class ConflictError(Exception):
    pass

store = CheckpointStore()
v = store.save("r1", {"step": 1})
print("ver", v, store.load("r1"))
try:
    store.save("r1", {"step": 2}, expected_version=0)
except ConflictError as e:
    print("conflict", e)
print("ok", store.save("r1", {"step": 2}, expected_version=1))


In [ ]:
# Demo 3: resume loop
STEPS = ["ingest", "retrieve", "draft", "send"]

def run_from(state):
    i = state.get("cursor", 0)
    while i < len(STEPS):
        state["log"] = state.get("log", []) + [STEPS[i]]
        i += 1
        state["cursor"] = i
        if STEPS[i - 1] == "draft":
            state["status"] = "waiting_human"
            return state  # checkpoint boundary
    state["status"] = "succeeded"
    return state

st = run_from({"cursor": 0})
print("paused", st)
st["flags"] = {"human_approved": True}
st["status"] = "running"
st = run_from(st)
print("final", st)


## Schema Evolution

### Definition
Changing the shape of state over time while old checkpoints still load.

### Why it matters
Ship velocity dies if every change breaks in-flight runs.

### How it works
Add `schema_version`; write migrations `v_n → v_{n+1}`; prefer additive fields; never reuse field meanings.

### Intuition
Expand/contract pattern from DB migrations, applied to agent state.

### Pitfalls
- Silent defaults that change behavior
- Deleting fields still needed by old code paths
- No tests on old fixtures

### When to use
As soon as you have a second production deploy.


In [ ]:
# Demo 4: migrate state forward
def migrate(state):
    v = state.get("schema_version", 1)
    if v == 1:
        state = {**state, "schema_version": 2, "flags": state.get("flags", {}), "messages": state.get("messages", [])}
        v = 2
    if v == 2:
        flags = dict(state.get("flags", {}))
        flags.setdefault("human_approved", False)
        state = {**state, "schema_version": 3, "flags": flags}
    return state

old = {"schema_version": 1, "messages": [{"role": "user", "content": "hi"}]}
print(migrate(old))


In [ ]:
# Demo 5: redact PII before checkpoint persistence
import re

def redact_state(state: dict) -> dict:
    s = deepcopy(state)
    for m in s.get("messages", []):
        m["content"] = re.sub(r"\b\d{3}-\d{2}-\d{4}\b", "[SSN]", m["content"])
    return s

from copy import deepcopy
raw = {"messages": [{"role": "user", "content": "SSN 123-45-6789"}]}
print(redact_state(raw))


### Try it yourself — State

1. Add a `fencing_token` so two workers cannot checkpoint the same run.
2. Write migrate tests for fixtures at schema 1,2,3.
3. Decide: should tool raw responses live in state? Write a 5-line policy.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `checkpoint` | Persisted snapshot of run state |
| `cursor` | Pointer to next step |
| `schema_version` | Integer identifying state shape |
| `optimistic locking` | Detect concurrent updates via version |


## State Size Budgeting

| Field class | Soft limit | Strategy |
|-------------|------------|----------|
| messages | last 20 or 4k tokens | summarize older |
| tool traces | last N / pointers | cold object storage |
| binary | never inline | URI only |
| secrets | never | secret manager ref |

### Fencing
Only the worker holding `lease_owner` + matching `fence` may checkpoint.


In [ ]:
# Lease / fence
class LeasedRun:
    def __init__(self):
        self.fence = 0
        self.owner = None
    def acquire(self, worker):
        self.fence += 1
        self.owner = worker
        return self.fence
    def checkpoint(self, worker, fence, state):
        if worker != self.owner or fence != self.fence:
            raise PermissionError("lost lease")
        return {"ok": True, "state": state}

r = LeasedRun()
f = r.acquire("w1")
print(r.checkpoint("w1", f, {"x": 1}))
r.acquire("w2")
try:
    r.checkpoint("w1", f, {"x": 2})
except PermissionError as e:
    print(e)


In [ ]:
# Expand/contract migration
# v1: {"user": "a"}
# v2: {"user_id": "a"}  (expand: write both; contract: stop reading old)

def to_v2(state):
    s = dict(state)
    if "user" in s and "user_id" not in s:
        s["user_id"] = s["user"]
    s["schema_version"] = 2
    return s

def contract_v2(state):
    s = dict(state)
    s.pop("user", None)
    return s

print(contract_v2(to_v2({"user": "a", "schema_version": 1})))


### Try it yourself — State deepen

1. Implement message summarization when messages > 10.
2. Store a large 'pdf' as URI and assert checkpoint JSON < 2KB.


## Key Takeaways

- State holds working set + pointers + flags
- Checkpoint at safe boundaries
- Migrate schemas explicitly
- Minimize PII in checkpoints
